In [11]:
import pandas as pd

df = pd.read_csv('../data/data.csv', sep=';')
df.columns = df.columns.str.strip()

In [12]:
df.shape

(4424, 37)

The dataset has 4,424 students (rows) and 37 columns: 36 features plus the Target outcome.

In [13]:
df.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


Each row is one student. The columns cover background, family, money, and grades for semesters 1 and 2.

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance                      4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualification                          4424

Every column is a number except Target, which is text. Categories like Course, Marital status, and parents' education are stored as number codes (for example, Course 9500 is one program), so they are labels, not amounts. I'll need to treat them as categories when modeling.

In [15]:
df['Target'].value_counts()

Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

Outcomes: 2,209 Graduate, 1,421 Dropout, 794 Enrolled.

In [16]:
df['Target'].value_counts(normalize=True).round(3)

Target
Graduate    0.499
Dropout     0.321
Enrolled    0.179
Name: proportion, dtype: float64

About 32% of students dropped out, 50% graduated, and 18% were still enrolled. The data is unbalanced: dropouts are about 1 in 3. So accuracy alone can be misleading. I should also check how many real dropouts the model catches (recall).

In [17]:
df.isna().sum().sum()

np.int64(0)

There are no missing values, so no cleaning is needed for blanks.

In [18]:
df['dropout'] = (df['Target'] == 'Dropout').astype(int)
df['dropout'].value_counts()

dropout
0    3003
1    1421
Name: count, dtype: int64

I made a new column, dropout: 1 = dropped out (1,421 students), 0 = did not (3,003 students, graduated or still enrolled). This is what the model will predict, like a real early warning tool.


Why fairness matters

**1. What Wisconsin's system did**
Since 2012, Wisconsin has used a machine learning tool called the Dropout Early Warning System (DEWS) to predict whether middle schoolers will graduate from high school on time. Twice a year it labels 6th to 9th graders green, yellow, or red for risk. It uses test scores, attendance, discipline records, and how often a student changes schools. It also uses race, gender, and family income.

**2. What went wrong, and who it hurt**
The Markup (2023) found that the "high risk" label was usually wrong: about 74% of students it predicted would not graduate actually did. The false alarms were not spread evenly. The false alarm rate was 42 percentage points higher for Black students and 18 points higher for Hispanic students than for White students. So students of color were much more likely to be labeled "at risk" when they were fine. That label can lower how teachers see a student and how students see themselves. Most teachers didn't know race was part of the model, and a UC Berkeley study found no evidence that DEWS improved graduation rates. The tool added risk for students without clear benefit.

**3. How my project could make the same mistake**
My data has no race column, but it has other columns that can act the same way: age at enrollment, gender, scholarship status, debt, being international, and parents' education (first-generation). If my model leans on these, it could give more false alarms to older students, first-gen students, or students with money trouble, even when their grades are fine. It could also miss students in some groups more often. To avoid repeating Wisconsin's mistakes, I will:
- Check false alarm and miss rates for each group, not just overall accuracy.
- Choose the flagging cutoff based on the cost of mistakes, then recheck fairness at that cutoff.
- Explain each prediction with SHAP, so advisors can see why a student was flagged.
- Keep a human in the loop: the model suggests who to talk to, an advisor decides, and students are never told they are "high risk."